In [11]:
document=[ 
    # Chủ đề: Công nghệ & AI
    "ChromaDB là một cơ sở dữ liệu vector mã nguồn mở được thiết kế chuyên biệt để lưu trữ và truy vấn các embedding cho ứng dụng AI và LLM.",
    "RAG (Retrieval-Augmented Generation) giúp mô hình ngôn ngữ lớn giảm thiểu ảo giác bằng cách truy xuất dữ liệu thực tế từ cơ sở dữ liệu bên ngoài trước khi trả lời.",
    "Python là ngôn ngữ lập trình phổ biến nhất hiện nay trong lĩnh vực Khoa học dữ liệu, Học máy và phát triển các hệ thống Backend.",
    
    # Chủ đề: Ẩm thực Việt Nam
    "Phở bò Hà Nội nổi tiếng với nước dùng trong vắt, ngọt thanh từ xương hầm, kết hợp cùng bánh phở mềm, thịt bò tái chín và hành lá.",
    "Bánh mì Sài Gòn là sự giao thoa ẩm thực độc đáo với lớp vỏ giòn rụm, nhân pate béo ngậy, chả lụa, thịt nguội, đồ chua và rau ngò.",
    "Cà phê sữa đá Việt Nam sử dụng hạt Robusta đậm vị, pha bằng phin truyền thống và kết hợp cùng sữa đặc có đường.",
    
    # Chủ đề: Du lịch & Địa lý
    "Vịnh Hạ Long thuộc tỉnh Quảng Ninh, được UNESCO công nhận là Di sản Thiên nhiên Thế giới với hàng nghìn đảo đá vôi kỳ vĩ trên mặt nước xanh ngọc.",
    "Đà Lạt nằm trên cao nguyên Lâm Viên, có khí hậu mát mẻ quanh năm, nổi tiếng với rừng thông, thác nước và kiến trúc biệt thự Pháp cổ.",
    "Phố cổ Hội An từng là thương cảng quốc tế sầm uất vào thế kỷ 17 và 18, nổi bật với những ngôi nhà tường vàng mái ngói rêu phong và đèn lồng rực rỡ."
]

In [15]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
load_dotenv()

class LangChainGeminiEF(EmbeddingFunction):
    def __init__(self):
        self.ef = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    def __call__(self, input: Documents) -> Embeddings:
        return self.ef.embed_documents(input)
gemini_ef=LangChainGeminiEF()

client=chromadb.Client()
try:
    client.delete_collection(name="new_baocao")
except Exception:
    pass
Collection=client.create_collection(
    name="new_baocao",
    embedding_function=gemini_ef
) 

In [16]:

Collection.add( 
    documents=document,
    
metadatas = [
    {"category": "tech", "lang": "vi", "source": "doc_1"},
    {"category": "tech", "lang": "vi", "source": "doc_2"},
    {"category": "tech", "lang": "vi", "source": "doc_3"},
    {"category": "food", "lang": "vi", "source": "doc_4"},
    {"category": "food", "lang": "vi", "source": "doc_5"},
    {"category": "food", "lang": "vi", "source": "doc_6"},
    {"category": "travel", "lang": "vi", "source": "doc_7"},
    {"category": "travel", "lang": "vi", "source": "doc_8"},
    {"category": "travel", "lang": "vi", "source": "doc_9"}
],
ids = [f"id_{i}" for i in range(1, len(document) + 1)])




In [17]:
data= Collection.get(
    include=["documents","embeddings","metadatas"])
if data["embeddings"] is not None and len(data["embeddings"]) > 0:
 for i,eb in enumerate (data['embeddings']):
    print(f"\nDocument {i+1}")
    print("Text: ", data["documents"][i])
    print("Embedding length: ",len(eb))
    print("First 10 values: ",eb[:10])
else: 
    print("Collection hiện đang trống")
    
    


Document 1
Text:  ChromaDB là một cơ sở dữ liệu vector mã nguồn mở được thiết kế chuyên biệt để lưu trữ và truy vấn các embedding cho ứng dụng AI và LLM.
Embedding length:  3072
First 10 values:  [-0.0334223  -0.00439302 -0.01454431 -0.07074519 -0.00735685  0.01228965
  0.00235088  0.00982695  0.01400127 -0.0204835 ]

Document 2
Text:  RAG (Retrieval-Augmented Generation) giúp mô hình ngôn ngữ lớn giảm thiểu ảo giác bằng cách truy xuất dữ liệu thực tế từ cơ sở dữ liệu bên ngoài trước khi trả lời.
Embedding length:  3072
First 10 values:  [-0.02029143 -0.00316754  0.01020507 -0.05901104 -0.01238899  0.01082738
  0.00399021 -0.00012856 -0.00178797  0.00342957]

Document 3
Text:  Python là ngôn ngữ lập trình phổ biến nhất hiện nay trong lĩnh vực Khoa học dữ liệu, Học máy và phát triển các hệ thống Backend.
Embedding length:  3072
First 10 values:  [-0.0157312   0.01522968  0.00036918 -0.05681124 -0.02062372  0.0203007
 -0.01019507 -0.00124061  0.00613406 -0.00595111]

Document 4
Text:  P

In [21]:
result=Collection.query(
    query_texts=['đây là 1 câu truy vấn liên quan tới danh lam thắng cảnh của Việt Nam cực kỳ nổi tiếng về vẻ đẹp hùng vĩ cũng như cổ kính'],
    n_results=2)
for i in range(len(result['documents'][0])):
    print(f"Top {i+1} (Distance: {result['distances'][0][i]:.4f}):")
    print("Nội dung:", result['documents'][0][i])

Top 1 (Distance: 0.3983):
Nội dung: Vịnh Hạ Long thuộc tỉnh Quảng Ninh, được UNESCO công nhận là Di sản Thiên nhiên Thế giới với hàng nghìn đảo đá vôi kỳ vĩ trên mặt nước xanh ngọc.
Top 2 (Distance: 0.4373):
Nội dung: Đà Lạt nằm trên cao nguyên Lâm Viên, có khí hậu mát mẻ quanh năm, nổi tiếng với rừng thông, thác nước và kiến trúc biệt thự Pháp cổ.
